# 토큰 — LLM 이 글자를 세는 방식

LLM 은 글자나 단어가 아니라 **토큰(token)** 단위로 글을 읽고 씁니다.
그리고 **요금도 토큰 단위로 매겨집니다.** 그래서 토큰을 알아야 비용을 가늠할 수 있습니다.

## 토큰이란

```
"하늘은 왜 파란가요"  ->  [3131, 44635, 4740, 120367, 36349, 43514, 4081, 7952]
                            9글자                    8개 토큰
```

자주 나오는 덩어리에 번호를 붙여 둔 **사전**이라고 보면 됩니다.
모델은 이 번호만 봅니다.

## 이 노트북의 흐름

| 단계 | 내용 |
|---|---|
| Step 1~3 | OpenAI 의 토큰 세기 (`tiktoken`) |
| Step 4~5 | Claude·Gemini 는 같은 문장을 몇 개로 셀까 |
| Step 6~9 | Ollama 로 **한국어와 영어 비교** |

> **미리 말해 두는 결론**
> 같은 문장인데 모델마다 세는 수가 다릅니다. **서로 비교하면 안 됩니다.**

## Step 1. `tiktoken` — OpenAI 의 토큰 계산기

`tiktoken` 은 OpenAI 가 공개한 라이브러리입니다.
**API 를 호출하지 않고** 내 PC 에서 토큰 수를 셀 수 있습니다.

> 호출 전에 미리 세어 볼 수 있다는 점이 중요합니다.
> 긴 문서를 보내기 전에 **요금과 길이 제한을 미리 확인**할 수 있습니다.

In [2]:
import tiktoken
from dotenv import load_dotenv


## Step 2. 비교할 문장 하나

```python
text = "하늘은 왜 파란가요"
```

공백 포함 9글자입니다. 이 한 문장을 **네 가지 방식**으로 세어 비교합니다.

In [3]:
load_dotenv(override=True)

text = "하늘은 왜 파란가요"

## Step 3-1. 세고 되돌리는 함수

```python
tokens = encoding.encode(input_text)   # 글자 -> 번호 목록
decoded_text = encoding.decode(tokens) # 번호 목록 -> 글자
```

`encode` 와 `decode` 는 **서로 반대 동작**입니다.
되돌린 글자가 원래 문장과 같은지 확인하려고 `decode` 까지 해 봅니다.

In [4]:
def token_id_count_decoded(input_text, encoding):
    tokens = encoding.encode(input_text)
    print("Token IDs:", tokens)
    print("Token count:", len(tokens))

    # Decode tokens back into text
    decoded_text = encoding.decode(tokens)
    print("Decoded:", decoded_text)

## Step 3-2. GPT-4o 기준 — 8 토큰

```
Token IDs: [3131, 44635, 4740, 120367, 36349, 43514, 4081, 7952]
Token count: 8
```

9글자가 **8개 토큰**이 됐습니다. 번호가 제각각인 것을 보세요.
`120367` 처럼 큰 번호는 **사전 뒤쪽에 있는 덜 흔한 덩어리**라는 뜻입니다.

### 한국어는 영어보다 토큰을 더 쓴다

`tiktoken` 으로 직접 재 본 결과입니다 (gpt-4o 기준).

| 한국어 | 토큰 | 영어 | 토큰 | 배율 |
|---|---|---|---|---|
| 하늘은 왜 파란가요 | 8 | Why is the sky blue | 5 | 1.6배 |
| 안녕하세요, 만나뵈서 반갑습니다. | 12 | Hello, nice to meet you. | 7 | 1.7배 |

**같은 뜻인데 한국어가 약 1.6배** 듭니다. 영어 위주로 사전이 만들어졌기 때문입니다.
그만큼 요금도 더 나옵니다.

In [5]:
# openai token count
encoding = tiktoken.encoding_for_model("gpt-4o")
token_id_count_decoded(text, encoding)


Token IDs: [3131, 44635, 4740, 120367, 36349, 43514, 4081, 7952]
Token count: 8
Decoded: 하늘은 왜 파란가요


## Step 4. Claude 는 18 토큰?

```
18
```

GPT-4o 는 8 인데 Claude 는 **18** 입니다. 두 배가 넘습니다.
**토크나이저가 나빠서가 아닙니다.** 세는 대상이 다릅니다.

| | 무엇을 셌나 |
|---|---|
| `tiktoken` | **글자만** |
| Claude `count_tokens` | 글자 + **메시지 형식**(role 표시 등) |

Claude 는 `messages=[{"role": "user", ...}]` 전체를 세므로
대화 형식에 들어가는 토큰이 함께 잡힙니다.

> **교훈: 회사끼리 토큰 수를 직접 비교하면 안 됩니다.**
> 측정 방법이 다르면 숫자도 다릅니다. 비용을 비교하려면
> 토큰 수가 아니라 **"같은 작업에 실제로 든 요금"** 을 봐야 합니다.

In [9]:
# claude token count
import anthropic

client = anthropic.Anthropic()

# OpenAI·Gemini와 같은 문장(text)으로 비교합니다.
# Claude는 메시지 형식에 필요한 토큰이 함께 계산되어 순수 텍스트보다 조금 많게 나옵니다.
response = client.messages.count_tokens(
    model="claude-opus-5",
    messages=[{"role": "user", "content": text}],
)
print(response.input_tokens)

TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

## Step 5. Gemini 는 9 토큰

```
Total Tokens: 9
```

GPT-4o 8, Gemini 9, Claude 18 — **같은 문장, 세 가지 숫자**입니다.

| 모델 | 토큰 | 비고 |
|---|---|---|
| GPT-4o | 8 | 순수 텍스트 |
| Gemini 2.5 Flash | 9 | 순수 텍스트 |
| Claude | 18 | 메시지 형식 포함 |

순수 텍스트끼리인 GPT-4o 와 Gemini 는 8 대 9 로 비슷합니다.
**토크나이저가 회사마다 다르게 만들어졌기 때문**에 완전히 같을 수는 없습니다.

In [10]:
from google import genai

client = genai.Client()

# Call count_tokens on your model and contents
response = client.models.count_tokens(
    model="gemini-3.8-flash",
    contents=text,
)

print("Total Tokens:", response.total_tokens)

ModuleNotFoundError: No module named 'google'

## Step 6. Ollama 는 다른 방법으로 — 실제 사용량 보기

지금까지는 **보내기 전에 미리** 세어 봤습니다.
이제는 **실제로 호출하고 나서** 사용량을 받아 봅니다.

```python
url = "http://localhost:11434/api/generate"
```

`1_3_ollama` 에서 쓴 `/v1`(OpenAI 호환) 이 아니라 **Ollama 고유 API** 입니다.
토큰 통계를 더 자세히 주기 때문에 여기서는 이쪽을 씁니다.

In [11]:
import requests

# Ollama local API endpoint
url = "http://localhost:11434/api/generate"

## Step 7. 한국어 질문 준비

```python
payload = {
    "model": "llama3.2",
    "prompt": "하늘은 왜 파란가요?",
    "stream": False,
    "options": {"num_predict": 512}
}
```

| 항목 | 뜻 |
|---|---|
| `stream: False` | 답을 **다 만든 뒤 한 번에** 받는다. 통계를 바로 보려고 |
| `num_predict: 512` | 최대 출력 토큰 수. **모델이 같은 말을 반복해도 여기서 멈춘다** |

> 작은 모델은 같은 문장을 끝없이 반복하는 일이 있습니다.
> 상한을 두지 않으면 응답이 돌아오지 않을 수 있습니다.

In [12]:
# Request payload (stream=False gives final stats immediately)
payload = {
    "model": "llama3.2",
    "prompt": "하늘은 왜 파란가요?",
    "stream": False,
    "options": {"num_predict": 512}  # 최대 출력 토큰 수 (모델이 반복에 빠져도 여기서 멈춤)
}

## Step 8. 한국어 결과

```
Prompt Tokens: 33
Output Tokens: 365
Total Tokens: 398
```

| 항목 | 응답 필드 | 뜻 |
|---|---|---|
| 입력 | `prompt_eval_count` | 질문에 든 토큰 |
| 출력 | `eval_count` | 답에 든 토큰 |

### 여기서 꼭 볼 것: 출력이 입력의 11배

```
입력  33
출력 365   <-- 11배
```

**비용은 대부분 출력에서 납니다.** 게다가 유료 API 는 보통 출력 토큰이 입력보다 비쌉니다.
비용을 줄이려면 질문을 줄이는 것보다 **답을 짧게 시키는 편**이 효과적입니다.

> `"답은 세 문장 이내로"` 같은 한 줄이 요금을 크게 줄입니다.

답 내용도 읽어 보세요. 산소가 하늘을 파랗게 한다는 **틀린 설명**을 합니다.
(실제로는 질소·산소 분자에 의한 **레일리 산란** 때문입니다.)
작은 모델은 이렇게 자신 있게 틀립니다.

In [13]:
response = requests.post(url, json=payload)

if response.status_code == 200:
    data = response.json()
    
    # Extract token counts and durations from Ollama's response
    prompt_tokens = data.get("prompt_eval_count", 0)
    output_tokens = data.get("eval_count", 0)
    total_tokens = prompt_tokens + output_tokens
    
    print(f"Response: {data.get('response')}")
    print("\n--- Token Statistics ---")
    print(f"Prompt Tokens: {prompt_tokens}")
    print(f"Output Tokens: {output_tokens}")
    print(f"Total Tokens: {total_tokens}")
else:
    print(f"Error: {response.status_code}, {response.text}")

Response: 하늘은 파란색으로 보이는 이유는 우리 시선으로부터 지구의 방사선에 대한 반사가 발생하여 주는 때문입니다.

지구는 지구의 바다와 대기 중의 방사선의 반사에 의해 파란색으로 보입니다. 방사선은 지구의 바다와 대기 중에서 발생하는 전자와 양전자들이 반사하는 것입니다.

지구는 대기 중에서 방사선이 발생하는 이유는 방사선이 지구의 바다와 대기 중에서 발생하는 전자와 양전자들이 반사하는 것입니다. 이 전자와 양전자들은 주로 태양의 bức임rays로 인해 발생합니다. 태양의 bức임rays는 태양의 핵에서 발생하는 고温과 고Pressure가 인자로 인하여 발생합니다. 

태양의 bức임rays는 지구의 바다와 대기 중에서 방사선으로 변환되는 데 필요한 에너지와 물질을 제공합니다. 이 방사선은 지구의 바다와 대기 중에서 발생하는 전자와 양전자들이 반사하는 것입니다. 이 반사는 우리의 시선으로부터 파란색으로 보이는 이유입니다.

지구의 바다와 대기 중에서 발생하는 방사선은 매우 가벼운 방사선이기 때문에 우리가 보는 방사선의 강도는 매우 약합니다. 그러나 지구의 바다와 대기 중에서 발생하는 방사선의 반사는 우리의 시선으로부터 파란색으로 보이는 이유입니다.

이러한 이유로 하늘은 파란색으로 보이게 됩니다.

--- Token Statistics ---
Prompt Tokens: 33
Output Tokens: 358
Total Tokens: 391


## Step 9. 같은 질문을 영어로

```python
"prompt": "Why is the sky blue?"
```

내용은 같고 **언어만 영어**입니다. 토큰 수가 어떻게 달라지는지 봅니다.

In [14]:
payload2 = {
    "model": "llama3.2",
    "prompt": "Why is the sky blue?",
    "stream": False,
    "options": {"num_predict": 512}  # 최대 출력 토큰 수
}

## Step 10. 한국어 vs 영어 — 실제 결과

| | 한국어 | 영어 |
|---|---|---|
| 입력 토큰 | 33 | 31 |
| 출력 토큰 | 365 | 441 |

### 예상과 다른 결과

입력은 **33 대 31 로 거의 같습니다.** "한국어가 토큰을 훨씬 많이 먹는다"는 말과 다릅니다.
이유는 두 가지입니다.

1. **문장이 너무 짧습니다.** 대화 형식에 붙는 고정 토큰이 더 커서 차이가 묻힙니다
2. **모델마다 토크나이저가 다릅니다.** Step 3 의 1.6배는 **gpt-4o 기준**이지 llama3.2 가 아닙니다

> **교훈: 일반론을 그대로 믿지 말고 내 모델·내 데이터로 재 보세요.**
> 토큰 수는 모델과 문장에 따라 달라집니다.

출력 품질은 확실히 다릅니다. 영어 답은 레일리 산란을 **정확히** 설명합니다.
Step 8 의 한국어 답과 비교해 보세요. **같은 모델인데 언어에 따라 정확도가 다릅니다.**

In [15]:
response = requests.post(url, json=payload2)

if response.status_code == 200:
    data = response.json()
    
    # Extract token counts and durations from Ollama's response
    prompt_tokens = data.get("prompt_eval_count", 0)
    output_tokens = data.get("eval_count", 0)
    total_tokens = prompt_tokens + output_tokens
    
    print(f"Response: {data.get('response')}")
    print("\n--- Token Statistics ---")
    print(f"Prompt Tokens: {prompt_tokens}")
    print(f"Output Tokens: {output_tokens}")
    print(f"Total Tokens: {total_tokens}")
else:
    print(f"Error: {response.status_code}, {response.text}")

Response: The sky appears blue because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century.

Here's what happens:

1. **Sunlight enters the Earth's atmosphere**: When sunlight enters the Earth's atmosphere, it encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2).
2. **Scattering occurs**: These tiny molecules scatter the light in all directions, but they scatter shorter (blue) wavelengths more than longer (red) wavelengths. This is because the smaller molecules are more efficient at scattering the shorter wavelengths.
3. **Blue light is scattered**: As a result of this scattering, the blue light is distributed throughout the atmosphere, giving the sky its blue appearance.
4. **Red light continues straight**: Meanwhile, the longer wavelengths of light (like red and orange) continue to travel in a more direct path, reaching our eyes from a more direct angle.

The combined effect o

---

## 정리

| | 내용 |
|---|---|
| **토큰은 글자가 아니다** | 자주 나오는 덩어리에 붙인 번호다 |
| **모델마다 다르게 센다** | 같은 문장이 8 / 9 / 18 — 회사끼리 비교 금지 |
| **출력이 비용을 지배한다** | 입력 33 vs 출력 365. 답을 짧게 시키는 게 효과적 |
| **미리 셀 수 있다** | `tiktoken` 은 호출 없이 센다 |
| **직접 재 보라** | 일반론보다 내 모델·내 데이터의 실측값 |

### 직접 해 보기

1. 긴 한국어 문단으로 Step 3 을 다시 해 보세요. 영어와의 배율이 1.6배에 가까워지나요?
2. `num_predict` 를 `50` 으로 줄여 보세요. 출력 토큰과 `finish_reason` 이 어떻게 되나요?
3. 자주 쓰는 프롬프트의 토큰 수를 재 보고, 1,000번 호출하면 얼마인지 계산해 보세요.

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `1_6_stateless` | 모델이 기억하지 못한다는 것과 그 해결 |
| `1_7_context_openai` | 토큰 한도 안에서 대화를 관리하는 방법 |